# H_VisualConsistency — 시각적 구매 일관성

Stratum-adjusted customer-average Recall; both original grouping and evaluation cells survived. Customer-level cohort files stay in Drive. Rebuild them with REBUILD_GROUPS=True if needed.

**탐색적 분석.** Uploaded notebook cells were reorganized after the experiment. Saved CSV/JSON/bootstrap arrays are unchanged. Outputs and execution counts have been cleared. The core is pinned to the recorded experiment commit; dependency versions may still differ.


In [ ]:
HYPOTHESIS = "H_VisualConsistency"
# Fresh Colab runtime: run this setup once per notebook.
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from google.colab import drive

DRIVE_ROOT = "/content/drive/MyDrive/msrs_paper (1)"  # Change for your shortcut.
drive.mount("/content/drive")

CORE_COMMIT = "e6b73dc4a8aee549b4310cf4f3266e373b8632ca"
CORE_DIR = Path("/content/msrs_kwangsuk_core")
REPO_URL = "https://github.com/3sunghynix/multimodal-recsys.git"
if any(name.startswith("msrs_paper") for name in sys.modules):
    raise RuntimeError("Restart the Colab session before running setup again.")
if not CORE_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CORE_DIR)], check=True)
    subprocess.run(["git", "-C", str(CORE_DIR), "checkout", "--detach", CORE_COMMIT], check=True)
actual_commit = subprocess.check_output(
    ["git", "-C", str(CORE_DIR), "rev-parse", "HEAD"], text=True
).strip()
if actual_commit != CORE_COMMIT:
    raise RuntimeError("The existing core checkout has a different commit; use a fresh runtime.")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(CORE_DIR / "paper/requirements.txt")
], check=True)
sys.path.insert(0, str(CORE_DIR / "paper"))
from msrs_paper.config import load_all, Paths
from msrs_paper import metrics
from msrs_paper.utils import verify_manifest
cfg, exps, plan = load_all()
paths = Paths(DRIVE_ROOT)  # Do not create a missing shared-data folder.
assert (paths.root / "manifest.json").is_file(), "Check DRIVE_ROOT and the shared shortcut."
manifest = verify_manifest(paths)
print("Verified files:", len(manifest["files"]))
print("Core commit:", actual_commit)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_OUTPUT_ROOT = Path("/content/drive/MyDrive/kwangsuk_reproductions") / stamp / HYPOTHESIS

def historical_folder(suffix, timestamp):
    candidates = [
        p / timestamp for p in Path("/content/drive/MyDrive").iterdir()
        if p.is_dir() and p.name.endswith(suffix) and (p / timestamp).is_dir()
    ]
    if len(candidates) != 1:
        raise RuntimeError(f"Set the historical input folder explicitly: {candidates}")
    return candidates[0]


In [ ]:
# Default: re-score the saved customer groups.
REBUILD_GROUPS = False
if REBUILD_GROUPS:
    COHORT_ROOT = RUN_OUTPUT_ROOT / "cohorts"
else:
    COHORT_ROOT = historical_folder("visual_consistency", "20261014T082339075594Z")
print("Customer-group inputs:", COHORT_ROOT)


## 1. 고객 그룹
Past purchases only; recent 50 distinct products; dominant product type; history-size strata and within-stratum similarity thirds.

In [ ]:
if REBUILD_GROUPS:
    import json
    import numpy as np
    import pandas as pd

    from pathlib import Path
    from datetime import datetime, timezone

    root = Path(DRIVE_ROOT)

    items = pd.read_parquet(root / "processed/items.parquet")
    history = pd.read_parquet(root / "processed/main/history.parquet")

    with open(root / "processed/main/stats.json", encoding="utf-8") as f:
        split_stats = json.load(f)

    required_columns = {"user", "item_idx", "t_dat"}
    assert required_columns.issubset(history.columns), (
        f"Unexpected history columns: {history.columns.tolist()}"
    )

    # Use the same evaluation-customer population as existing predictions.
    prediction_users = np.load(
        root / "outputs/A/main/C2_seed42/users.npy",
        allow_pickle=False
    )

    # Strictly exclude purchases from the test week onward.
    test_start = pd.Timestamp(split_stats["test_start"])
    history["t_dat"] = pd.to_datetime(history["t_dat"])

    past = history.loc[
        (history["t_dat"] < test_start)
        & history["user"].isin(prediction_users),
        ["user", "item_idx", "t_dat"]
    ].copy()

    # Keep the most recent purchase of each distinct product,
    # then the customer's most recent 50 distinct products.
    past = past.sort_values(
        ["user", "t_dat", "item_idx"],
        ascending=[True, False, True],
        kind="mergesort"
    )
    past = past.drop_duplicates(["user", "item_idx"], keep="first")
    past = past.groupby("user", sort=False).head(50).copy()

    assert items["idx"].is_unique
    type_map = items.set_index("idx")["product_type_name"]
    past["product_type"] = past["item_idx"].map(type_map)
    past = past.dropna(subset=["product_type"])

    # Select each customer's most frequently purchased product type.
    # Alphabetical order resolves ties consistently.
    type_counts = (
        past.groupby(["user", "product_type"])
        .size()
        .rename("n_type_history")
        .reset_index()
    )

    dominant_types = (
        type_counts.sort_values(
            ["user", "n_type_history", "product_type"],
            ascending=[True, False, True]
        )
        .drop_duplicates("user")
    )

    # Require five distinct products in that type.
    dominant_types = dominant_types.loc[
        dominant_types["n_type_history"] >= 5
    ].copy()

    selected_history = past.merge(
        dominant_types[["user", "product_type"]],
        on=["user", "product_type"],
        how="inner"
    )

    image_vectors = np.load(
        root / "embeddings/img_marqo.npy",
        mmap_mode="r"
    )

    assert np.isfinite(image_vectors).all()

    rows = []

    for (user, product_type), group in selected_history.groupby(
        ["user", "product_type"], sort=False
    ):
        indices = group["item_idx"].to_numpy(dtype=int)
        vectors = np.asarray(image_vectors[indices], dtype=np.float64)

        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        assert np.all(norms > 0), "A zero-length image vector was found."
        vectors = vectors / norms

        n = len(vectors)

        # Average cosine similarity over every distinct pair.
        # This formula avoids constructing a large pairwise matrix.
        summed = vectors.sum(axis=0)
        consistency = (summed @ summed - n) / (n * (n - 1))

        rows.append({
            "user": int(user),
            "product_type": product_type,
            "n_type_history": n,
            "visual_consistency": float(
                np.clip(consistency, -1.0, 1.0)
            )
        })

    customers = pd.DataFrame(rows)
    assert len(customers) > 0, "No customers meet the history requirement."

    # Compare customers within the same product type and history-size band.
    customers["history_band"] = pd.cut(
        customers["n_type_history"],
        bins=[4, 9, 19, 50],
        labels=["5-9", "10-19", "20-50"],
        include_lowest=True
    ).astype(str)

    customers["consistency_group"] = "middle"
    threshold_rows = []

    MIN_GROUP_USERS = 30

    for (product_type, band), group in customers.groupby(
        ["product_type", "history_band"], sort=True
    ):
        q_low, q_high = group["visual_consistency"].quantile(
            [1 / 3, 2 / 3]
        ).to_numpy()

        # Skip strata that cannot form two distinct groups.
        if q_low >= q_high:
            continue

        low_indices = group.index[
            group["visual_consistency"] <= q_low
        ]
        high_indices = group.index[
            group["visual_consistency"] >= q_high
        ]

        if (
            len(low_indices) < MIN_GROUP_USERS
            or len(high_indices) < MIN_GROUP_USERS
        ):
            continue

        customers.loc[low_indices, "consistency_group"] = "low"
        customers.loc[high_indices, "consistency_group"] = "high"

        threshold_rows.append({
            "product_type": product_type,
            "history_band": band,
            "low_threshold": float(q_low),
            "high_threshold": float(q_high),
            "low_users": len(low_indices),
            "high_users": len(high_indices)
        })

    cohorts = customers.loc[
        customers["consistency_group"].isin(["low", "high"])
    ].copy()

    assert len(cohorts) > 0, (
        "No comparison groups meet the minimum customer requirement."
    )

    thresholds = pd.DataFrame(threshold_rows)

    # Save the grouping rules before scoring recommendation results.
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    # COHORT_ROOT is configured above.
    COHORT_ROOT.mkdir(parents=True, exist_ok=False)

    customers.to_csv(COHORT_ROOT / "all_customer_scores.csv", index=False)
    cohorts.to_csv(COHORT_ROOT / "customer_groups.csv", index=False)
    thresholds.to_csv(COHORT_ROOT / "group_thresholds.csv", index=False)

    protocol = {
        "hypothesis_id": "H_VisualConsistency",
        "status": "exploratory; existing main split",
        "hypothesis": (
            "Image contribution is greater for high visual-consistency customers."
        ),
        "history": (
            "Before test_start; most recent 50 distinct products;"
            " dominant product type with at least 5 distinct products."
        ),
        "consistency_measure": (
            "Mean pairwise cosine similarity of Marqo image embeddings."
        ),
        "grouping": (
            "Bottom and top thirds within product type and history-size band;"
            " minimum 30 customers per group."
        ),
        "evaluation_plan": (
            "Non-repurchase test purchases in the customer's dominant product type;"
            " customer-average Recall@300; C2 minus C4;"
            " compare high minus low within strata."
        ),
        "training_seeds": [42, 43, 44],
        "bootstrap": {"unit": "customer", "samples": 1000, "seed": 42},
        "test_start": str(test_start)
    }

    with open(COHORT_ROOT / "analysis_plan.json", "w", encoding="utf-8") as f:
        json.dump(protocol, f, ensure_ascii=False, indent=2)

    print("Customers with sufficient same-type history:", len(customers))
    print("\nSelected customer groups:")
    print(cohorts["consistency_group"].value_counts().to_string())

    print("\nEligible product-type/history strata:", len(thresholds))
    print(thresholds.to_string(index=False))

    print("\nMean visual consistency by group:")
    print(
        cohorts.groupby("consistency_group")["visual_consistency"]
        .mean()
        .to_string()
    )

    print("\nSaved to:", COHORT_ROOT)
else:
    print('Using saved customer groups; reconstruction skipped.')


## 2. 재채점
Evaluate non-repurchase purchases in the dominant type. Minimum 30 evaluated customers per group; paired real/shuffled scores and stratified resampling.

In [ ]:
import json
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from msrs_paper import metrics

# COHORT_ROOT is configured above.
root = Path(DRIVE_ROOT)

SEEDS = [42, 43, 44]
K = 300
N_BOOT = 1000
BOOT_SEED = 42
MIN_EVALUATED_USERS = 30

cohorts = pd.read_csv(COHORT_ROOT / "customer_groups.csv")
items = pd.read_parquet(root / "processed/items.parquet")
targets = pd.read_parquet(root / "processed/main/test_targets.parquet")

assert cohorts["user"].is_unique
assert items["idx"].is_unique

# Evaluate purchases in each customer's dominant product type.
type_map = items.set_index("idx")["product_type_name"]
targets["target_type"] = targets["item_idx"].map(type_map)
assert targets["target_type"].notna().all()

eligible_targets = targets.loc[
    ~targets["is_repurchase"]
].merge(
    cohorts[
        ["user", "product_type", "history_band", "consistency_group"]
    ],
    on="user",
    how="inner",
    validate="many_to_one"
)

eligible_targets = eligible_targets.loc[
    eligible_targets["target_type"].eq(
        eligible_targets["product_type"]
    )
].copy()

assert len(eligible_targets) > 0, "No qualifying test purchases."

# Identify customers who have at least one qualifying purchase.
evaluated_customers = cohorts.loc[
    cohorts["user"].isin(eligible_targets["user"])
].copy()

counts = (
    evaluated_customers.groupby(
        ["product_type", "history_band", "consistency_group"]
    ).size().unstack(fill_value=0)
    .reindex(columns=["low", "high"], fill_value=0)
)

print("Customers with qualifying purchases:")
print(counts.to_string())

keep = (
    (counts["low"] >= MIN_EVALUATED_USERS)
    & (counts["high"] >= MIN_EVALUATED_USERS)
)
eligible_strata = counts.index[keep].tolist()
assert eligible_strata, "No strata meet the evaluation-size requirement."

# Keep the same eligible strata for every model and seed.
valid_keys = set(eligible_strata)

evaluated_customers = evaluated_customers.loc[
    [
        (ptype, band) in valid_keys
        for ptype, band in zip(
            evaluated_customers["product_type"],
            evaluated_customers["history_band"]
        )
    ]
].sort_values("user").reset_index(drop=True)

eligible_targets = eligible_targets.loc[
    eligible_targets["user"].isin(evaluated_customers["user"])
].reset_index(drop=True)

users_order = evaluated_customers["user"].to_numpy(dtype=np.int64)
rows = metrics.target_rows(users_order, eligible_targets)

purchases_per_user = np.bincount(
    rows, minlength=len(users_order)
)
assert np.all(purchases_per_user > 0)

# Fixed weights: each stratum's share of evaluated customers,
# combining high and low groups.
selected_counts = counts.loc[eligible_strata].copy()
weights = selected_counts.sum(axis=1).astype(float)
weights = weights / weights.sum()

# Record the evaluation rules before reading prediction scores.
with open(COHORT_ROOT / "analysis_plan.json", encoding="utf-8") as f:
    protocol = json.load(f)

protocol["evaluation_details"] = {
    "min_evaluated_users_per_group": MIN_EVALUATED_USERS,
    "strata_weight": "Share of evaluated customers, high and low combined",
    "metric": "Mean of customer-level Recall@300 within each stratum/group",
    "bootstrap": (
        "Resample customers within each stratum and consistency group;"
        " pair real and shuffled results for each sampled customer"
    )
}

RUN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
evaluation_plan_file = RUN_OUTPUT_ROOT / "evaluation_plan.json"
if evaluation_plan_file.exists():
    with open(evaluation_plan_file, encoding="utf-8") as f:
        previous = json.load(f)
    assert previous == protocol, "Evaluation rules changed."
else:
    with open(evaluation_plan_file, "w", encoding="utf-8") as f:
        json.dump(protocol, f, ensure_ascii=False, indent=2)

print("\nIncluded strata:", len(eligible_strata))
print("Evaluated customers:", len(users_order))
print("Evaluated purchases:", len(eligible_targets))

def load_customer_recall(condition, seed):
    folder = (
        root / "outputs/A/main"
        / f"{condition}_seed{seed}"
    )

    prediction_users = np.load(
        folder / "users.npy", allow_pickle=False
    )
    candidates = np.load(
        folder / "candidates.npy",
        mmap_mode="r",
        allow_pickle=False
    )

    assert prediction_users.ndim == 1
    assert len(np.unique(prediction_users)) == len(prediction_users)
    assert candidates.shape[0] == len(prediction_users)
    assert candidates.shape[1] >= K

    prediction_rows = metrics.target_rows(
        prediction_users, eligible_targets
    )
    hits = metrics.hits_at_k(
        candidates,
        prediction_rows,
        eligible_targets["item_idx"].to_numpy(),
        K
    ).astype(np.int64)

    hits_per_user = np.bincount(
        rows, weights=hits, minlength=len(users_order)
    )
    return hits_per_user / purchases_per_user

real_by_seed = []
shuffled_by_seed = []

for seed in SEEDS:
    print(f"Scoring seed {seed}...")
    real_by_seed.append(load_customer_recall("C2", seed))
    shuffled_by_seed.append(load_customer_recall("C4", seed))

real_by_seed = np.stack(real_by_seed)
shuffled_by_seed = np.stack(shuffled_by_seed)

real_mean = real_by_seed.mean(axis=0)
shuffled_mean = shuffled_by_seed.mean(axis=0)
contribution = real_mean - shuffled_mean

# Customer positions for each matched stratum and group.
group_positions = {}
table_rows = []

for product_type, band in eligible_strata:
    for group in ["low", "high"]:
        positions = np.flatnonzero(
            evaluated_customers["product_type"].eq(product_type)
            & evaluated_customers["history_band"].eq(band)
            & evaluated_customers["consistency_group"].eq(group)
        )
        group_positions[(product_type, band, group)] = positions

        table_rows.append({
            "product_type": product_type,
            "history_band": band,
            "consistency_group": group,
            "customers": int(len(positions)),
            "purchases": int(purchases_per_user[positions].sum()),
            "real_recall_at_300": float(real_mean[positions].mean()),
            "shuffled_recall_at_300": float(
                shuffled_mean[positions].mean()
            ),
            "image_contribution": float(
                contribution[positions].mean()
            )
        })

def adjusted_group_mean(values, group):
    return float(sum(
        weights.loc[(ptype, band)]
        * values[group_positions[(ptype, band, group)]].mean()
        for ptype, band in eligible_strata
    ))

low_contribution = adjusted_group_mean(contribution, "low")
high_contribution = adjusted_group_mean(contribution, "high")
estimate = high_contribution - low_contribution

# Stratified customer bootstrap; real/shuffled scores stay paired.
rng = np.random.default_rng(BOOT_SEED)
bootstrap_estimates = np.zeros(N_BOOT)

for (ptype, band), weight in weights.items():
    for group, sign in [("high", 1), ("low", -1)]:
        values = contribution[
            group_positions[(ptype, band, group)]
        ]

        # Batches keep memory use modest.
        for start in range(0, N_BOOT, 50):
            stop = min(start + 50, N_BOOT)
            sampled = rng.integers(
                0, len(values),
                size=(stop - start, len(values))
            )
            bootstrap_estimates[start:stop] += (
                sign * float(weight) * values[sampled].mean(axis=1)
            )

low_ci, high_ci = np.quantile(
    bootstrap_estimates, [0.025, 0.975]
)

seed_rows = []
for i, seed in enumerate(SEEDS):
    seed_contribution = real_by_seed[i] - shuffled_by_seed[i]
    low_value = adjusted_group_mean(seed_contribution, "low")
    high_value = adjusted_group_mean(seed_contribution, "high")

    seed_rows.append({
        "seed": seed,
        "low_contribution": low_value,
        "high_contribution": high_value,
        "difference_high_minus_low": high_value - low_value
    })

group_summary = pd.DataFrame([
    {
        "consistency_group": group,
        "adjusted_real_recall": adjusted_group_mean(real_mean, group),
        "adjusted_shuffled_recall": adjusted_group_mean(
            shuffled_mean, group
        ),
        "adjusted_image_contribution": adjusted_group_mean(
            contribution, group
        )
    }
    for group in ["low", "high"]
])

stratum_table = pd.DataFrame(table_rows)
seed_table = pd.DataFrame(seed_rows)

summary = {
    "hypothesis_id": "H_VisualConsistency",
    "status": "exploratory",
    "measure": "Stratum-adjusted customer-average Recall@300",
    "contrast": "High minus low image contribution; C2 minus C4",
    "n_strata": len(eligible_strata),
    "n_evaluated_customers": len(users_order),
    "n_evaluated_purchases": len(eligible_targets),
    "low_contribution": low_contribution,
    "high_contribution": high_contribution,
    "difference": estimate,
    "ci95_customer_bootstrap": [float(low_ci), float(high_ci)],
    "seeds": SEEDS,
    "bootstrap_samples": N_BOOT,
    "bootstrap_seed": BOOT_SEED
}

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
result_dir = RUN_OUTPUT_ROOT / "results"
result_dir.mkdir(parents=True, exist_ok=False)

counts.to_csv(result_dir / "evaluation_customer_counts.csv")
weights.rename("weight").to_csv(result_dir / "stratum_weights.csv")
stratum_table.to_csv(result_dir / "stratum_scores.csv", index=False)
group_summary.to_csv(result_dir / "group_summary.csv", index=False)
seed_table.to_csv(result_dir / "seed_scores.csv", index=False)
np.save(result_dir / "bootstrap_estimates.npy", bootstrap_estimates)

with open(result_dir / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nAdjusted group results:")
print(group_summary.round(6).to_string(index=False))

print("\nPer-seed results:")
print(seed_table.round(6).to_string(index=False))

print(f"\nHigh minus low contribution: {estimate:+.6f}")
print(
    f"95% customer-bootstrap interval:"
    f" [{low_ci:+.6f}, {high_ci:+.6f}]"
)
print("\nSaved to:", result_dir)